In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

groq_model = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")

@tool
def weather_tool(city:str)-> str:
    """get the wheter of the specified city"""   # very important
    return f"the weather of {city} is rainy."  # instead of this we can call any weather api

groq_model_with_tools = groq_model.bind_tools([weather_tool])

response = groq_model_with_tools.invoke("How is the weather in Boston city ?")

print(response)
for tool_call in response.tool_calls:
    print(tool_call['name'])
    print(tool_call["args"])

content='' additional_kwargs={'tool_calls': [{'id': 'jd47v7h1z', 'function': {'arguments': '{"city":"Boston"}', 'name': 'whether_tool'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 224, 'total_tokens': 239, 'completion_time': 0.024914857, 'completion_tokens_details': None, 'prompt_time': 0.012412929, 'prompt_tokens_details': None, 'queue_time': 0.058224735, 'total_time': 0.037327786}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f9a5f-01e6-7df3-a9e5-0961d9190835-0' tool_calls=[{'name': 'whether_tool', 'args': {'city': 'Boston'}, 'id': 'jd47v7h1z', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 224, 'output_tokens': 15, 'total_tokens': 239}
whether_tool
{'city': 'Boston'}


`model response`  
```
content='' 
additional_kwargs={'tool_calls': [{'id': '3pza4t2qv', 'function': {'arguments': '{"city":"boston"}', 'name': 'whether_tool'}, 'type': 'function'}]} 
response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 224, 'total_tokens': 240, 'completion_time': 0.035568626, 'completion_tokens_details': None, 'prompt_time': 0.014135733, 'prompt_tokens_details': None, 'queue_time': 0.055936616, 'total_time': 0.049704359}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} 
id='lc_run--019f9975-ad67-76d0-9046-ce96de5ac201-0' 
tool_calls=[{'name': 'whether_tool', 'args': {'city': 'boston'}, 'id': '3pza4t2qv', 'type': 'tool_call'}] 
invalid_tool_calls=[] 
usage_metadata={'input_tokens': 224, 'output_tokens': 16, 'total_tokens': 240}
```

In [5]:
## google
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

google_model = ChatGoogleGenerativeAI(
    model = "gemini-3.6-flash"
)

@tool
def weather_tool(city:str)->str:
    """get the Weather of a city"""
    return f"the weather in {city} is rainy."

google_model_with_tools = google_model.bind_tools([weather_tool])

response = google_model_with_tools.invoke("What is the weather in Boston city.")
print(response)

content=[] additional_kwargs={'function_call': {'name': 'weather_tool', 'arguments': '{"city": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'h4EW31L8': 'EpwCCpkCARFNMg/7lwOyMPAPcY0OBb2uX/ykDKmgpOoNzGY7X3P5GDtvMO8zzrEfOuFCmoFzwS5E3cVe7rm0lMPaVLcRxBrpatNBzrM4PoG/z2rzz8BJnbDeKTmbQQs/xD9+MriN4sPbbk+yGTZoJ+Dd4hH6yeTdVtdfEH52xAyDHUgVfbMr5PIuPFaUZz4oQkW1EontzUwO1q79oCO2bqaqsGOyk14QcEfPSgzn32igjUfN7l+GoMeLAchSS1Mpihfnv4ud0YQAvLCbOcO8t3a00R/9KCx7i4kriH560G94ohE3toac8pIv88qhULQ8aBdGLEiEtfv2fMyV7PC7WV0NfCD4qdwGXBX9z6rcnkK9ysE92ZfrkjnxsjbIhxE='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f9a6a-d5f9-7343-adcf-ee0a18bb50f1-0' tool_calls=[{'name': 'weather_tool', 'args': {'city': 'Boston'}, 'id': 'h4EW31L8', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 52, 'output_tokens': 78, 'total_tokens': 130, 'input_token_details': {'cache_read': 0}, 'output_to

In [7]:
## google
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

google_model = ChatGoogleGenerativeAI(
    model = "gemini-3.6-flash"
)

@tool
def weather_tool(city:str)->str:
    """get the Weather of a city"""
    return f"the weather in {city} is rainy."

google_model_with_tools = google_model.bind_tools([weather_tool])

# 2. First invocation triggers the tool call request
first_response = google_model_with_tools.invoke("What is the weather in Boston city.")

# 3. Extract and execute the tool manually
tool_calls = first_response.tool_calls
if tool_calls:
    selected_tool = tool_calls[0]
    # Execute tool function dynamically
    tool_output = weather_tool.invoke(selected_tool["args"])
    
    # Create a ToolMessage to feed back into the model
    tool_message = ToolMessage(
        content=str(tool_output), 
        tool_call_id=selected_tool["id"]
    )
    
    # 4. Second invocation gives the final natural language answer
    final_response = google_model_with_tools.invoke([first_response, tool_message])
    print(final_response.content)  
    # Output: "The weather in Boston is rainy."

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.6-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Please ensure that function call turn comes immediately after a user turn or after a function response turn.', 'status': 'INVALID_ARGUMENT'}}

### Tool Execution Loop

In [9]:
## groq

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

groq_model = ChatGroq(model="llama-3.3-70b-versatile")

@tool 
def groq_weather_tool(city:str)-> str:
    """get the weather of an city"""
    return f"the weather in {city} is rainy."

groq_model_with_tools = groq_model.bind_tools([groq_weather_tool])

first_response = groq_model_with_tools.invoke("What is the weather in Pune?")
# print(first_response)

groq_tool_call = first_response.tool_calls

if groq_tool_call:
    selected_tool = groq_tool_call[0]
    tool_response = groq_weather_tool.invoke(selected_tool['args'])
    toolMessage = ToolMessage(content= str(tool_response), tool_call_id=selected_tool['id'])
    
final_response = groq_model_with_tools.invoke([first_response, toolMessage])
print("REsponse ",final_response.content)

REsponse  


This is a very common stumbling block when working with LangChain's tool calling! The reason your final_response.content is empty is that you are missing the original user prompt in your final LLM invocation.When you pass [first_response, toolMessage] to the model, you are only giving it the AI's tool request and the tool's output. Because the model doesn't see the original question ("What is the weather in Pune?"), it lacks the context needed to formulate a final, human-readable answer.To fix this, you need to pass the full conversation history (User Question $\rightarrow$ AI Tool Call $\rightarrow$ Tool Result) back to the model.

In [13]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage, HumanMessage

groq_model = ChatGroq(model="llama-3.3-70b-versatile")

@tool 
def groq_weather_tool(city:str)-> str:
    """get the weather of an city"""
    return f"the weather in {city} is rainy."

groq_model_with_tools = groq_model.bind_tools([groq_weather_tool])

# 1. Start a message history list with the user's prompt
messages = [HumanMessage(content="What is the weather in Pune?")]

# 2. Pass the history to the model
first_response = groq_model_with_tools.invoke(messages)

# 3. Append the AI's response (which contains the tool call) to the history
messages.append(first_response)

groq_tool_call = first_response.tool_calls

if groq_tool_call:
    selected_tool = groq_tool_call[0]
    tool_response = groq_weather_tool.invoke(selected_tool['args'])
    toolMessage = ToolMessage(content=str(tool_response), tool_call_id=selected_tool['id'])
    
    # 4. Append the tool's result to the history
    messages.append(toolMessage)
    
# 5. Invoke the model again with the FULL context
final_response = groq_model_with_tools.invoke(messages)
print(final_response.text)

In [23]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage, HumanMessage

groq_model = ChatGroq(model="llama-3.3-70b-versatile")

@tool 
def groq_weather_tool(city: str) -> str:
    """
    Gets the current weather for a specific city. 
    Returns a string describing the weather. 
    Once you receive this data, immediately answer the user's question without calling this tool again.
    """
    return f"the weather in {city} is rainy."

# Bind the tool to the model
groq_model_with_tools = groq_model.bind_tools([groq_weather_tool])

# 1. Initialize message history
messages = [HumanMessage(content="What is the weather in Pune?")]

# 2. Get first response and append it
first_response = groq_model_with_tools.invoke(messages)
messages.append(first_response)

groq_tool_call = first_response.tool_calls

if groq_tool_call:
    selected_tool = groq_tool_call[0]
    
    # Invoke the actual Python function
    tool_response = groq_weather_tool.invoke(selected_tool['args'])
    
    # 3. Create ToolMessage WITH the required 'name' parameter
    toolMessage = ToolMessage(
        content=str(tool_response), 
        tool_call_id=selected_tool['id'],
        name=selected_tool['name'] # <-- CRITICAL FOR GROQ/LLAMA 3
    )
    
    messages.append(toolMessage)
    
# 4. Get final response
final_response = groq_model_with_tools.invoke(messages)

# 5. Debugging check to see what the model actually returned
if final_response.tool_calls:
    print("⚠️ The model made ANOTHER tool call instead of answering:")
    print(final_response.tool_calls)
    print(messages)
else:
    print("✅ Final Answer:")
    print(final_response.content)

✅ Final Answer:
The weather in Pune is rainy.


2. The "Double Tool Call" Loop
Sometimes, instead of giving you a final text answer, Llama 3 decides it needs to call the tool again. In LangChain, whenever an AIMessage contains tool calls, its text .content attribute is intentionally left empty. If your model is stuck in a loop, you will see an empty .content but a populated .tool_calls list on the final response.

In [21]:
## groq

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

groq_model = ChatGroq(model="llama-3.3-70b-versatile")

@tool 
def groq_weather_tool(city:str)-> str:
    """
    Gets the current weather for a specific city. 
    Returns a string describing the weather. 
    Once you receive this data, immediately answer the user's question without calling this tool again.
    """
    return f"the weather in {city} is rainy."

groq_model_with_tools = groq_model.bind_tools([groq_weather_tool])
messages = [HumanMessage("What is the weather in Pune?")]

first_response = groq_model_with_tools.invoke(messages)
# print(first_response)
messages.append(first_response)

groq_tool_call = first_response.tool_calls

if groq_tool_call:
    selected_tool = groq_tool_call[0]
    tool_response = groq_weather_tool.invoke(selected_tool['args'])
    toolMessage = ToolMessage(content= str(tool_response), tool_call_id=selected_tool['id'], name = selected_tool['name'])
messages.append(toolMessage)

final_response = groq_model_with_tools.invoke(messages)
print("REsponse ",final_response.content)

REsponse  The weather in Pune is rainy.
